# Mandarin (en-zh) fine-tune on Colab GPU

Before running: **Runtime -> Change runtime type -> Hardware accelerator -> GPU** (T4 or whatever's available) -> Save.

No GitHub access needed -- `train.py` has no local imports, so this notebook
writes it and its config directly to the Colab VM instead of cloning a repo.

**One-time manual step, do this first**: upload the 4 processed data files
(`data/processed/en-zh/{train,val}.{en,zh}`, ~22MB total, from the local repo)
to a Google Drive folder, e.g. `MyDrive/medical_vlm_data/en-zh/`. Update
`DRIVE_DATA_DIR` below to match wherever you put them.

Checkpoints are written to Drive (not the ephemeral Colab VM disk), so if
the session disconnects (free tier: ~90 min idle / ~12h max), just
reconnect and re-run this notebook top to bottom -- the last cell
auto-detects an existing checkpoint and resumes from it instead of
starting over. Every cell that depends on the working directory sets it
explicitly, so re-running just a subset of cells after a disconnect (not
strictly top-to-bottom) also works.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- stop and pick a GPU runtime')
assert torch.cuda.is_available(), 'No GPU attached -- use Runtime -> Change runtime type -> GPU'

CUDA available: True
Device: Tesla T4


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

name, memory.total [MiB], memory.used [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 3 MiB, 14910 MiB


In [ ]:
import os

# No GitHub needed: train.py has zero local imports (only external packages),
# so the next two cells just write it and the config straight to disk instead
# of cloning a repo -- avoids the private-repo auth problem entirely.
REPO_DIR = "/content/repo"
os.makedirs(f"{REPO_DIR}/src", exist_ok=True)
os.makedirs(f"{REPO_DIR}/configs", exist_ok=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

cwd: /content/repo


In [ ]:
%%writefile src/train.py
"""Fine-tune a pretrained MarianMT checkpoint on a medical parallel corpus.

Mirrors teammate 2's per-direction MarianMT fine-tuning pattern: one pretrained
Helsinki-NLP/opus-mt-* checkpoint in, one fine-tuned checkpoint out, driven by
a small YAML config so the same script covers every direction (en-zh, zh-en,
en-hi, hi-en, ...) with only the config changed.

Usage:
    python src/train.py --config configs/en-zh.yaml
"""
import argparse
from pathlib import Path

import numpy as np
import torch
import yaml
from datasets import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


def load_parallel_file_pair(src_path: Path, tgt_path: Path) -> Dataset:
    with src_path.open(encoding="utf-8") as f:
        src_lines = [l.rstrip("\n") for l in f]
    with tgt_path.open(encoding="utf-8") as f:
        tgt_lines = [l.rstrip("\n") for l in f]
    assert len(src_lines) == len(tgt_lines), (
        f"{src_path} has {len(src_lines)} lines, {tgt_path} has {len(tgt_lines)}"
    )
    return Dataset.from_dict({"src": src_lines, "tgt": tgt_lines})


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", required=True, type=Path)
    ap.add_argument("--resume-from-checkpoint", default=None)
    args = ap.parse_args()

    cfg = yaml.safe_load(args.config.read_text(encoding="utf-8"))

    base_model = cfg["base_model"]
    output_dir = Path(cfg["output_dir"])
    max_src_len = cfg.get("max_source_length", 128)
    max_tgt_len = cfg.get("max_target_length", 128)

    print(f"Loading base model {base_model} ...")
    tokenizer = MarianTokenizer.from_pretrained(base_model)
    model = MarianMTModel.from_pretrained(base_model)

    train_ds = load_parallel_file_pair(Path(cfg["train_src"]), Path(cfg["train_tgt"]))
    val_ds = load_parallel_file_pair(Path(cfg["val_src"]), Path(cfg["val_tgt"]))
    print(f"Train examples: {len(train_ds)} | Val examples: {len(val_ds)}")

    def preprocess(batch):
        model_inputs = tokenizer(
            batch["src"], max_length=max_src_len, truncation=True,
        )
        labels = tokenizer(
            text_target=batch["tgt"], max_length=max_tgt_len, truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
    val_tok = val_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    def compute_metrics(eval_preds):
        import sacrebleu

        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
        return {"bleu": bleu.score}

    training_args_kwargs = dict(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=cfg.get("learning_rate", 2e-5),
        per_device_train_batch_size=cfg.get("per_device_train_batch_size", 16),
        per_device_eval_batch_size=cfg.get("per_device_eval_batch_size", 16),
        weight_decay=cfg.get("weight_decay", 0.01),
        num_train_epochs=cfg.get("num_train_epochs", 3),
        predict_with_generate=True,
        generation_max_length=max_tgt_len,
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        save_total_limit=cfg.get("save_total_limit", 2),
        report_to=cfg.get("report_to", []),
        group_by_length=cfg.get("group_by_length", False),
        dataloader_num_workers=cfg.get("dataloader_num_workers", 0),
    )
    # Filter against whatever this installed transformers version actually
    # accepts -- API surface (e.g. field renames) drifts across major
    # versions, and this script needs to run unmodified on both a pinned
    # local install and whatever Colab happens to have pulled in.
    import dataclasses
    valid_fields = {f.name for f in dataclasses.fields(Seq2SeqTrainingArguments)}
    dropped = {k: v for k, v in training_args_kwargs.items() if k not in valid_fields}
    if dropped:
        print(f"NOTE: this transformers install doesn't support {list(dropped)} -- skipping (had no effect anyway if unsupported)")
    training_args_kwargs = {k: v for k, v in training_args_kwargs.items() if k in valid_fields}

    training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.get("early_stopping_patience", 3))],
    )

    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    final_dir = output_dir / "final"
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f"Saved fine-tuned model to {final_dir}")


if __name__ == "__main__":
    main()

Writing src/train.py


In [ ]:
%%writefile configs/en-zh-colab-gpu.yaml
# Same fine-tune as configs/en-zh.yaml, retuned for GPU throughput (Colab).
# Kept separate from the CPU config on purpose: batch size and dataloader
# settings are hardware-dependent tuning, not methodology, so the CPU run's
# config stays untouched. learning_rate is deliberately left unchanged from
# the CPU config, even though batch size is 4x larger -- scaling both at once
# would conflate a throughput change with an optimization-trajectory change,
# and this is a small enough model/LR that the standard linear-scaling
# argument for bumping LR with batch size doesn't clearly apply here.
#
# Data expected at data/processed/en-zh/{train,val}.{en,zh}
# (same pooled data as the CPU run -- copy from Drive, see colab_train_en_zh.ipynb)

base_model: Helsinki-NLP/opus-mt-en-zh
output_dir: models/opus-mt-en-zh-medical

train_src: data/processed/en-zh/train.en
train_tgt: data/processed/en-zh/train.zh
val_src: data/processed/en-zh/val.en
val_tgt: data/processed/en-zh/val.zh

max_source_length: 128
max_target_length: 128

num_train_epochs: 3
learning_rate: 2.0e-5

# 4x the CPU config's batch size -- this is a ~77M-param model at seq_len 128,
# comfortably small for any Colab GPU tier (T4 and up). If `!nvidia-smi` in
# the notebook shows headroom, this can go higher (e.g. 128) for more speed.
per_device_train_batch_size: 64
per_device_eval_batch_size: 64

# Buckets similar-length examples into the same batch, so batches pad to the
# length of their own examples instead of the longest in a random batch --
# real speedup (less wasted compute on padding tokens) with no quality cost.
group_by_length: true

# Colab typically gives ~2 CPU cores; overlaps batch collation with GPU compute
# instead of blocking on it (irrelevant on the CPU run, where dataloading was
# never the bottleneck -- GPU compute is fast enough that it can be).
dataloader_num_workers: 2

weight_decay: 0.01
early_stopping_patience: 3
save_total_limit: 2
report_to: []

Writing configs/en-zh-colab-gpu.yaml


In [ ]:
# Deliberately NOT reinstalling torch -- Colab's preinstalled torch already has
# CUDA wired up correctly; reinstalling from requirements.txt risks silently
# swapping it for a CPU-only build.
!pip install -q transformers sentencepiece sacrebleu PyYAML accelerate sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil

os.chdir("/content/repo")  # hardcoded so this cell is self-contained even after a partial re-run

DRIVE_DATA_DIR = "/content/drive/MyDrive/medical_vlm_data/en-zh"  # where you uploaded train/val.{en,zh}
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical"

assert os.path.isdir(DRIVE_DATA_DIR), f"{DRIVE_DATA_DIR} not found -- upload the processed data there first"
for f in ["train.en", "train.zh", "val.en", "val.zh"]:
    assert os.path.isfile(os.path.join(DRIVE_DATA_DIR, f)), f"missing {f} in {DRIVE_DATA_DIR}"

os.makedirs("data/processed/en-zh", exist_ok=True)
for f in ["train.en", "train.zh", "val.en", "val.zh"]:
    shutil.copy(os.path.join(DRIVE_DATA_DIR, f), os.path.join("data/processed/en-zh", f))

# models/ -> Drive, so checkpoints (and the final model) survive a disconnect
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
if os.path.islink("models") or os.path.isdir("models"):
    if os.path.islink("models"):
        os.remove("models")
    else:
        shutil.rmtree("models")
os.symlink("/content/drive/MyDrive/medical_vlm_checkpoints", "models")
print("Data and checkpoint dir ready. cwd:", os.getcwd())

Data and checkpoint dir ready. cwd: /content/repo


In [ ]:
import os, glob

os.chdir("/content/repo")  # hardcoded so this cell is self-contained even after a partial re-run

DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical"
checkpoints = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINT_DIR, "checkpoint-*")), key=os.path.getmtime)
resume_flag = f'--resume-from-checkpoint "{checkpoints[-1]}"' if checkpoints else ""
if checkpoints:
    print(f"Resuming from {checkpoints[-1]}")
else:
    print("No existing checkpoint found -- starting fresh.")

# en-zh-colab-gpu.yaml, not en-zh.yaml: same data/model/epochs, but batch size,
# group_by_length, and dataloader_num_workers are retuned for GPU throughput
# (see the config file's comments for why each change is safe).
!python src/train.py --config configs/en-zh-colab-gpu.yaml {resume_flag}

No existing checkpoint found -- starting fresh.
Loading base model Helsinki-NLP/opus-mt-en-zh ...
tokenizer_config.json: 100% 44.0/44.0 [00:00<00:00, 249kB/s]
source.spm: 100% 806k/806k [00:00<00:00, 17.3MB/s]
target.spm: 100% 805k/805k [00:00<00:00, 105MB/s]
vocab.json: 100% 1.62M/1.62M [00:00<00:00, 121MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
config.json: 100% 1.40k/1.40k [00:00<00:00, 1.82MB/s]

pytorch_model.bin: downloading bytes:  72% 224M/312M [00:01<00:00, 256MB/s, 18.9MB/s  ]
pytorch_model.bin: downloading bytes:  86% 268M/312M [00:01<00:00, 242MB/s, 23.7MB/s  ]
pytorch_model.bin: downloading bytes: 100% 294M/294M [00:02<00:00, 145MB/s, 27.1MB/s  ]
pytorch_model.bin: reconstructing file: 100% 312M/312M [00:02<00:00, 154MB/s, 29.5MB/s  ]
Loading weights: 100% 258/258 [00:00<00:00, 16214.12it/s]

model.safetensors: 

In [ ]:
!ls -la models/opus-mt-en-zh-medical/final
!readlink -f models/opus-mt-en-zh-medical

total 305991
-rw------- 1 root root      1414 Aug  3 21:51 config.json
-rw------- 1 root root       994 Aug  3 21:51 generation_config.json
-rw------- 1 root root 309965092 Aug  3 21:51 model.safetensors
-rw------- 1 root root    806435 Aug  3 21:51 source.spm
-rw------- 1 root root    804600 Aug  3 21:51 target.spm
-rw------- 1 root root       851 Aug  3 21:51 tokenizer_config.json
-rw------- 1 root root      5329 Aug  3 21:51 training_args.bin
-rw------- 1 root root   1747795 Aug  3 21:51 vocab.json
/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical


In [ ]:
import os, shutil
os.chdir("/content/repo")
DRIVE_EVAL_DIR = "/content/drive/MyDrive/medical_vlm_data/eval/nejm_zh"
assert os.path.isdir(DRIVE_EVAL_DIR), f"{DRIVE_EVAL_DIR} not found -- upload nejm.test.{{en,zh}} there first"
for f in ["nejm.test.en", "nejm.test.zh"]:
    assert os.path.isfile(os.path.join(DRIVE_EVAL_DIR, f)), f"missing {f} in {DRIVE_EVAL_DIR}"
os.makedirs("data/eval/nejm_zh", exist_ok=True)
for f in ["nejm.test.en", "nejm.test.zh"]:
    shutil.copy(os.path.join(DRIVE_EVAL_DIR, f), os.path.join("data/eval/nejm_zh", f))
print("Eval data ready:", os.listdir("data/eval/nejm_zh"))


In [ ]:
import re
import torch
from transformers import MarianMTModel, MarianTokenizer

def detok_en(t):
    t = t.replace(" @-@ ", "-")
    t = re.sub(r"\s+([,.;:!?)])", r"\1", t)
    t = re.sub(r"([(])\s+", r"\1", t)
    return t.strip()

def detok_zh(t):
    return t.replace(" ", "").strip()

with open("data/eval/nejm_zh/nejm.test.en", encoding="utf-8") as f:
    en_lines = f.read().splitlines()
with open("data/eval/nejm_zh/nejm.test.zh", encoding="utf-8") as f:
    zh_lines = f.read().splitlines()

check_indices = [922, 1778, 1959, 1960]  # 1959 = the "atelectasis" case the pretrained model got wrong

print(f"Loading fine-tuned checkpoint on {'cuda' if torch.cuda.is_available() else 'cpu'} ...")
tokenizer = MarianTokenizer.from_pretrained("models/opus-mt-en-zh-medical/final")
model = MarianMTModel.from_pretrained("models/opus-mt-en-zh-medical/final")
if torch.cuda.is_available():
    model = model.cuda()
model.eval()

for i in check_indices:
    en = detok_en(en_lines[i])
    ref = detok_zh(zh_lines[i])
    inputs = tokenizer([en], return_tensors="pt", truncation=True, max_length=128)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_length=128)
    hyp = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    print(f"\n[test_split_index {i}]")
    print("en: ", en)
    print("ref:", ref)
    print("hyp:", hyp)


In [ ]:
%%writefile src/baseline_eval.py
"""Baseline evaluation of a *pretrained, not-yet-fine-tuned* MarianMT checkpoint,
scored against real held-out human-translated data already in this project.

Why this script exists and runs before any fine-tuning: fine-tuning without a
baseline means there is nothing to compare against afterward -- you'd have a
model and a feeling that it's probably better, with no number to put in a
paper. This measures the pretrained checkpoint first, so `train.py`'s output
can later be measured with the exact same script and compared directly.

*** Standing rule: never generate ground truth ***
An earlier version of this script used hand-written pilot sentences with
reference translations invented by Claude. That was wrong and has been
replaced: every source/reference pair scored here is pulled verbatim from
real, existing, human-translated data -- never written or guessed on the
spot, even as a placeholder. Inventing a reference makes the eval circular
(you'd be measuring "how similar is this to what an AI guessed," not "how
correct is this translation"), which defeats the entire purpose of a number
meant to appear in a paper. Real sources used:
  - Mandarin: `data/eval/nejm_zh/nejm.test.{en,zh}` -- NEJM-enzh's own
    test split (2,102 pairs), medical, professionally human-translated.
    (Moved out of data/raw/en-zh/nejm_enzh/ into data/eval/ so data_prep.py's
    pooling can't accidentally sweep the held-out test split into training --
    it originally could, which would have made this baseline invalid once a
    fine-tuned model was compared against it.)
  - Hindi:    `data/eval/tico19_hi/tico19.{en,hi}` -- TICO-19's test split
    (2,100 pairs), medical/COVID-domain, human-translated.
Both are held-out test splits, never touched by data_prep.py's training
pool, so they stay valid for a fine-tuned-model re-run later too.

Since the pilot label set (cardiomegaly, pleural effusion, pneumothorax,
edema, "no acute cardiopulmonary abnormality", support devices,
atelectasis, consolidation) comes from chest-X-ray reporting (CheXpert/
MIMIC-CXR labels) and these two real corpora are general internal-medicine
literature / COVID guidance -- not radiology reports -- most of those exact
terms are rare or absent in the real data. Per the standing rule, that
absence is reported explicitly (see "pilot_term_coverage" in the output),
not papered over with invented sentences. Where a term *is* found, treat it
as a rough substring match, not a guaranteed topical match -- inspect the
matched sentence text in the report; e.g. "cardiopulmonary" mostly turns up
"cardiopulmonary resuscitation" in NEJM, not the radiology sense.

Known preprocessing fix applied here (per the spec's tokenization note):
NEJM-enzh's raw files are Moses-style pre-tokenized (English has `@-@`
hyphen-splitting and spaced punctuation; Chinese is word-segmented with
spaces between words, which is not how written Chinese normally looks).
Both are de-tokenized on the fly before translation/scoring below --
otherwise the model would be penalized for an artifact of how the file is
stored, not for actual translation quality.

Usage:
    python src/baseline_eval.py --lang zh --model Helsinki-NLP/opus-mt-en-zh --label baseline --out results/baseline_en-zh.json
    python src/baseline_eval.py --lang hi --model Helsinki-NLP/opus-mt-en-hi --label baseline --out results/baseline_en-hi.json

    # after fine-tuning, re-run against the fine-tuned checkpoint for a direct delta:
    python src/baseline_eval.py --lang zh --model models/opus-mt-en-zh-medical/final --label fine-tuned --out results/finetuned_en-zh.json
"""
import argparse
import json
import re
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_MODEL = {
    "zh": "Helsinki-NLP/opus-mt-en-zh",
    "hi": "Helsinki-NLP/opus-mt-en-hi",
}

DATA_SOURCES = {
    "zh": {
        "src": Path("data/eval/nejm_zh/nejm.test.en"),
        "tgt": Path("data/eval/nejm_zh/nejm.test.zh"),
        "description": "NEJM-enzh test split (2,102 pairs), medical, human-translated (professional NEJM translators)",
        "detokenize_src": True,   # Moses-style: @-@ splitting, spaced punctuation
        "detokenize_tgt": True,   # word-segmented Chinese -> strip spaces
    },
    "hi": {
        "src": Path("data/eval/tico19_hi/tico19.en"),
        "tgt": Path("data/eval/tico19_hi/tico19.hi"),
        "description": "TICO-19 en-hi test split (2,100 pairs), medical/COVID-domain, human-translated",
        "detokenize_src": False,
        "detokenize_tgt": False,
    },
}

# (label, substring) -- substring matching is a rough proxy, not a guaranteed
# topical match; the report includes the actual matched sentence so this can
# be sanity-checked by eye.
PILOT_TERMS = [
    ("cardiomegaly", "cardiomegaly"),
    ("pleural_effusion", "pleural effusion"),
    ("pneumothorax", "pneumothorax"),
    ("edema", "edema"),
    ("no_acute_cardiopulmonary_abnormality_EXACT_PHRASE", "no acute cardiopulmonary abnormality"),
    ("cardiopulmonary_BROAD_LOOSE_MATCH", "cardiopulmonary"),
    ("support_devices", "support device"),
    ("atelectasis", "atelectasis"),
    ("consolidation", "consolidation"),
]


def detokenize_moses_en(text: str) -> str:
    text = text.replace(" @-@ ", "-")
    text = re.sub(r"\s+([,.;:!?)])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    return text.strip()


def detokenize_segmented_zh(text: str) -> str:
    return text.replace(" ", "").strip()


def load_pairs(src_path: Path, tgt_path: Path, detok_src: bool, detok_tgt: bool):
    src_lines = src_path.read_text(encoding="utf-8").splitlines()
    tgt_lines = tgt_path.read_text(encoding="utf-8").splitlines()
    assert len(src_lines) == len(tgt_lines), f"{src_path} has {len(src_lines)} lines, {tgt_path} has {len(tgt_lines)}"
    pairs = []
    for s, t in zip(src_lines, tgt_lines):
        if detok_src:
            s = detokenize_moses_en(s)
        if detok_tgt:
            t = detokenize_segmented_zh(t)
        pairs.append((s, t))
    return pairs


def find_term_matches(pairs, terms):
    matches = {label: [] for label, _ in terms}
    for i, (en, _ref) in enumerate(pairs):
        low = en.lower()
        for label, substring in terms:
            if substring in low:
                matches[label].append(i)
    return matches


def cosine_sim(a, b) -> float:
    import numpy as np
    a = np.asarray(a, dtype="float64")
    b = np.asarray(b, dtype="float64")
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--lang", required=True, choices=sorted(DATA_SOURCES), help="target language")
    ap.add_argument("--model", default=None, help=f"checkpoint to evaluate (HF hub id or local dir). Defaults per --lang: {DEFAULT_MODEL}")
    ap.add_argument("--labse-model", default="sentence-transformers/LaBSE")
    ap.add_argument("--label", default="baseline", help="free-text tag for this run (e.g. 'baseline', 'fine-tuned'), stored in the report")
    ap.add_argument("--out", required=True, type=Path)
    ap.add_argument("--max-length", type=int, default=128)
    ap.add_argument("--sample-size", type=int, default=200, help="random sample size drawn from the non-pilot-matched remainder of the test split (0 = use the entire remainder, i.e. the full test split)")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    model_id = args.model or DEFAULT_MODEL[args.lang]
    source = DATA_SOURCES[args.lang]
    for key in ("src", "tgt"):
        if not source[key].exists():
            raise SystemExit(f"Expected real held-out data at {source[key]} but it doesn't exist.")

    print(f"Data source: {source['description']}")
    pairs = load_pairs(source["src"], source["tgt"], source["detokenize_src"], source["detokenize_tgt"])
    print(f"Loaded {len(pairs)} real held-out pairs from {source['src'].name} / {source['tgt'].name}")

    term_matches = find_term_matches(pairs, PILOT_TERMS)
    pilot_term_coverage = {label: len(idxs) for label, idxs in term_matches.items()}
    print("\nPilot term coverage in real held-out data (0 = absent, reported honestly, not filled in):")
    for label, count in pilot_term_coverage.items():
        print(f"  {label}: {count}")

    matched_indices = sorted({i for idxs in term_matches.values() for i in idxs})
    matched_set = set(matched_indices)
    remaining_indices = [i for i in range(len(pairs)) if i not in matched_set]

    import random
    rng = random.Random(args.seed)
    if args.sample_size and args.sample_size > 0:
        sample_indices = rng.sample(remaining_indices, min(args.sample_size, len(remaining_indices)))
    else:
        sample_indices = remaining_indices
    selected_indices = sorted(matched_set | set(sample_indices))
    print(f"\nEvaluating {len(selected_indices)} sentences total: {len(matched_indices)} pilot-term matches + {len(sample_indices)} randomly sampled (seed={args.seed}) from the remainder")

    index_to_terms = {}
    for label, idxs in term_matches.items():
        for i in idxs:
            index_to_terms.setdefault(i, []).append(label)

    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import MarianMTModel, MarianTokenizer
    import sacrebleu

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nDevice: {device}")
    print(f"Loading MT checkpoint: {model_id}")
    tokenizer = MarianTokenizer.from_pretrained(model_id)
    model = MarianMTModel.from_pretrained(model_id).to(device)
    model.eval()

    print(f"Loading LaBSE: {args.labse_model}")
    labse = SentenceTransformer(args.labse_model, device=device)

    per_sentence = []
    hypotheses, references = [], []

    for n, i in enumerate(selected_indices, 1):
        en, ref = pairs[i]
        terms_here = index_to_terms.get(i, [])

        inputs = tokenizer([en], return_tensors="pt", truncation=True, max_length=args.max_length).to(device)
        start = time.perf_counter()
        with torch.no_grad():
            generated = model.generate(**inputs, max_length=args.max_length)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        hyp = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

        emb = labse.encode([hyp, ref], normalize_embeddings=False)
        labse_score = cosine_sim(emb[0], emb[1])
        sent_bleu = sacrebleu.sentence_bleu(hyp, [ref]).score

        per_sentence.append({
            "test_split_index": i,
            "pilot_terms_matched": terms_here,
            "source_en": en,
            "reference": ref,
            "hypothesis": hyp,
            "labse_cosine": labse_score,
            "sentence_bleu": sent_bleu,
            "inference_seconds": elapsed,
        })
        hypotheses.append(hyp)
        references.append(ref)

        if n % 25 == 0 or n == len(selected_indices):
            print(f"  [{n}/{len(selected_indices)}] {elapsed*1000:.0f}ms  LaBSE={labse_score:.3f}  BLEU={sent_bleu:.1f}" + (f"  terms={terms_here}" if terms_here else ""))

    def summarize(rows):
        if not rows:
            return None
        labse_scores = [r["labse_cosine"] for r in rows]
        timings = [r["inference_seconds"] for r in rows]
        hyps = [r["hypothesis"] for r in rows]
        refs = [r["reference"] for r in rows]
        return {
            "n_sentences": len(rows),
            "mean_labse_cosine": statistics.mean(labse_scores),
            "min_labse_cosine": min(labse_scores),
            "corpus_bleu": sacrebleu.corpus_bleu(hyps, [refs]).score,
            "corpus_chrf": sacrebleu.corpus_chrf(hyps, [refs]).score,
            "mean_inference_seconds": statistics.mean(timings),
            "median_inference_seconds": statistics.median(timings),
            "max_inference_seconds": max(timings),
        }

    pilot_rows = [r for r in per_sentence if r["pilot_terms_matched"]]
    summary_overall = summarize(per_sentence)
    summary_pilot_subset = summarize(pilot_rows)

    report = {
        "label": args.label,
        "model": model_id,
        "lang_pair": f"en-{args.lang}",
        "device": device,
        "labse_model": args.labse_model,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "data_source": {
            "src_file": str(source["src"]),
            "tgt_file": str(source["tgt"]),
            "description": source["description"],
        },
        "reference_translation_provenance": "Real human-translated held-out test data, never generated or invented.",
        "pilot_term_coverage": pilot_term_coverage,
        "sample_size_requested": args.sample_size,
        "seed": args.seed,
        "per_sentence": per_sentence,
        "summary_overall": summary_overall,
        "summary_pilot_term_subset": summary_pilot_subset,
    }

    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\n=== Summary (overall, all evaluated sentences) ===")
    print(f"model:              {model_id}")
    print(f"n sentences:        {summary_overall['n_sentences']}")
    print(f"mean LaBSE cosine:  {summary_overall['mean_labse_cosine']:.3f}")
    print(f"corpus BLEU:        {summary_overall['corpus_bleu']:.1f}")
    print(f"corpus chrF:        {summary_overall['corpus_chrf']:.1f}")
    print(f"mean inference:     {summary_overall['mean_inference_seconds']*1000:.0f} ms/sentence")
    print(f"median inference:   {summary_overall['median_inference_seconds']*1000:.0f} ms/sentence")
    if summary_pilot_subset:
        print(f"\n=== Summary (pilot-term-matched subset only, n={summary_pilot_subset['n_sentences']}) ===")
        print(f"mean LaBSE cosine:  {summary_pilot_subset['mean_labse_cosine']:.3f}")
        print(f"corpus BLEU:        {summary_pilot_subset['corpus_bleu']:.1f}")
    else:
        print("\n(no pilot-term matches found in this real test split -- see pilot_term_coverage above)")
    print(f"\nReport saved to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
import os, shutil

os.chdir("/content/repo")

# Same --seed as the original baseline run (both default to 42), against the
# same real held-out file -> selects the identical 212 sentences both times,
# so this is a genuine paired before/after comparison, not just two separate
# random samples that happen to be the same size.
!python src/baseline_eval.py --lang zh --model models/opus-mt-en-zh-medical/final --label fine-tuned --out results/finetuned_en-zh.json

# Persist the report to Drive too, alongside the checkpoint.
os.makedirs("/content/drive/MyDrive/medical_vlm_checkpoints/results", exist_ok=True)
shutil.copy("results/finetuned_en-zh.json", "/content/drive/MyDrive/medical_vlm_checkpoints/results/finetuned_en-zh.json")
print("Report also copied to Drive.")


In [ ]:
import json

# Real numbers already measured earlier (results/baseline_en-zh.json, generated
# before any fine-tuning) -- transcribed here, not re-derived or guessed, so this
# print doesn't depend on that file also being present in this Colab session.
BASELINE = {
    "overall": {"n": 212, "labse": 0.8718048154543968, "bleu": 10.663627875184597, "chrf": 28.305876693097925},
    "pilot_subset": {"n": 12, "labse": 0.8605227870049942, "bleu": 3.743209069966981},
}

with open("results/finetuned_en-zh.json", encoding="utf-8") as f:
    finetuned = json.load(f)

fo, fp = finetuned["summary_overall"], finetuned["summary_pilot_term_subset"]

def fmt_delta(before, after):
    d = after - before
    return f"{before:.3f} -> {after:.3f}  ({'+' if d >= 0 else ''}{d:.3f})"

print("=== Overall (n=212 both runs, same seed -> same sentences) ===")
print("LaBSE: ", fmt_delta(BASELINE["overall"]["labse"], fo["mean_labse_cosine"]))
print("BLEU:  ", fmt_delta(BASELINE["overall"]["bleu"], fo["corpus_bleu"]))
print("chrF:  ", fmt_delta(BASELINE["overall"]["chrf"], fo["corpus_chrf"]))

if fp:
    print(f"\n=== Pilot-term-matched subset (n={fp['n_sentences']}, was n={BASELINE['pilot_subset']['n']} in baseline) ===")
    print("LaBSE: ", fmt_delta(BASELINE["pilot_subset"]["labse"], fp["mean_labse_cosine"]))
    print("BLEU:  ", fmt_delta(BASELINE["pilot_subset"]["bleu"], fp["corpus_bleu"]))
else:
    print("\n(no pilot-term matches in this run's subset -- unexpected if n differs from baseline's 12, check pilot_term_coverage)")
